<a href="https://colab.research.google.com/github/shreya777/Live-Meeting-Summarizer-Application/blob/main/week6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#  Install all dependencies
import subprocess, sys

# NumPy MUST be pinned first and alone before other packages
subprocess.run([sys.executable, "-m", "pip", "install",
                "numpy==1.26.4", "--force-reinstall", "-q"], check=True)

pkgs = [
    "pyannote.audio==4.0.4",
    "openai-whisper",
    "soundfile",
    "groq",
    "rouge-score",
    "jiwer",
]
for pkg in pkgs:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                      capture_output=True, text=True)
    print(f"{pkg}{'completed' if r.returncode==0 else 'not completed'} ")

print("\n All packages installed — now go to Runtime > Restart runtime")

pyannote.audio==4.0.4completed 
openai-whispercompleted 
soundfilecompleted 
groqcompleted 
rouge-scorecompleted 
jiwercompleted 

 All packages installed — now go to Runtime > Restart runtime


In [1]:
import os, sys, json, queue, wave, threading, logging, warnings
import numpy as np
import soundfile as sf
import whisper, torch
from pyannote.audio import Pipeline
from groq import Groq
from jiwer import wer, Compose, ToLowerCase, RemovePunctuation, RemoveMultipleSpaces, Strip
from google.colab import userdata

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

# Auth tokens
HF_TOKEN     = userdata.get('HF_TOKEN')
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# Config
AUDIO_PATH   = '/content/meeting2.wav'   # ← change to your file
NUM_SPEAKERS = 2                         # ← change as needed
OUTPUT_DIR   = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(' Imports OK')

 Imports OK


In [2]:
# Whisper STT (with word timestamps)
data, sr = sf.read(AUDIO_PATH)
duration = len(data) / sr
print(f' Audio: {duration:.1f}s = {int(duration//60)}m {int(duration%60)}s')

print('\n Running Whisper STT...')
stt_model    = whisper.load_model('base')
whisper_result = stt_model.transcribe(AUDIO_PATH, word_timestamps=True,
                                       language='en', fp16=False)
raw_transcript = whisper_result['text'].strip()

print(f' {len(whisper_result["segments"])} segments')
print(f'   Preview: {raw_transcript[:150]}...')

# Save raw transcript
with open(f'{OUTPUT_DIR}/raw_transcript.txt', 'w') as f:
    f.write(raw_transcript)
print(f' Saved → {OUTPUT_DIR}/raw_transcript.txt')

 Audio: 362.7s = 6m 2s

 Running Whisper STT...
 103 segments
   Preview: So according to the brief, we're going to be selling this remote control for 25 euros and we're aiming to make 50 million euro. So we're going to be s...
 Saved → /content/output/raw_transcript.txt


In [3]:

# Speaker Diarization
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print(' HF Login OK')

print('\n Loading pyannote pipeline...')
device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-3.1', token=HF_TOKEN
).to(device)
print(f' Pipeline ready (device={device})')

print(f'\n Diarizing (num_speakers={NUM_SPEAKERS})...')
out = pipeline(AUDIO_PATH, num_speakers=NUM_SPEAKERS)
ann = out.speaker_diarization if hasattr(out, 'speaker_diarization') else out

hyp = [(round(t.start,2), round(t.end,2), spk)
       for t,_,spk in ann.itertracks(yield_label=True)]

unique    = list(dict.fromkeys(s for _,_,s in hyp))
label_map = {sp: f'Speaker {i+1}' for i,sp in enumerate(unique)}
mapped_hyp = [{'start':s,'end':e,'speaker':label_map.get(sp,sp)} for s,e,sp in hyp]

print(f' Detected {len(unique)} speakers, {len(hyp)} turns')

# Align Whisper segments with diarization turns
lines = []
prev  = None
for seg in whisper_result['segments']:
    best_spk, best_ov = 'Unknown', 0
    for turn in mapped_hyp:
        ov = max(0, min(seg['end'],turn['end']) - max(seg['start'],turn['start']))
        if ov > best_ov:
            best_ov, best_spk = ov, turn['speaker']
    if best_spk != prev:
        lines.append(f'\n{best_spk}:')
        prev = best_spk
    lines.append(f"  {seg['text'].strip()}")

labelled_transcript = '\n'.join(lines)

print('\n SPEAKER-WISE TRANSCRIPT')
print('='*55)
print(labelled_transcript)
print('='*55)

with open(f'{OUTPUT_DIR}/diarized_transcript.txt', 'w') as f:
    f.write(labelled_transcript)
print(f'\n Saved → {OUTPUT_DIR}/diarized_transcript.txt')

✅ HF Login OK

🔍 Loading pyannote pipeline...
✅ Pipeline ready (device=cuda)

🔍 Diarizing (num_speakers=2)...
✅ Detected 2 speakers, 100 turns

📋 SPEAKER-WISE TRANSCRIPT

Speaker 1:
  So according to the brief, we're going to be selling this remote control for 25
  euros and we're aiming to make 50 million euro.
  So we're going to be selling this one in international scale and we don't want it to cost
  any more than 1250 euros so 50% of the selling price.

Speaker 2:
  Can we just go over that again?
  Sure.
  So, basically, 12.
  Alright, yeah.
  So, production cost is 1250 but selling price is that wholesale or retail, like on the shelf.

Speaker 1:
  I don't know, I imagine that's a good question.
  I imagine it probably is or sale actually because it's probably up to the retailer to sell
  it for whatever price they want.
  But I don't know.
  Do you think the fact that it's going to be sold internationally will have a bearing
  on how we design it at all?
  Yes.
  I think it wil

In [6]:
# Summarization (runs AFTER diarization)
client = Groq(api_key=GROQ_API_KEY)

def create_prompt(transcript):
    return f"""You are an AI assistant that summarizes meeting transcripts.

Analyze the following meeting transcript and generate a structured summary.

Transcript:
{transcript}

Provide the output in this format:

1. Key Points
2. Decisions Made
3. Action Items
4. Short Summary

Be clear and concise."""

def generate_summary(transcript):
    response = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[{'role': 'user', 'content': create_prompt(transcript)}],
        temperature=0.3
    )
    return response.choices[0].message.content

print(' Generating summary...')
summary = generate_summary(labelled_transcript)

print(' MEETING SUMMARY')
print(summary)

with open(f'{OUTPUT_DIR}/meeting_summary.txt', 'w') as f:
    f.write(summary)
print(f'\n Saved → {OUTPUT_DIR}/meeting_summary.txt')

 Generating summary...
 MEETING SUMMARY
**1. Key Points**

- The remote control will be sold for 25 euros, aiming to make 50 million euros.
- The production cost is estimated to be 1250 euros, which is 50% of the selling price.
- The remote control will be designed to be compatible with international markets, considering regional differences in languages, keypad styles, and symbols.
- The price of the remote control may vary depending on the market and consumer purchasing power.
- The remote control should be designed to be a premium product, with a unique and appealing design.
- The device should combine multiple functions, such as controlling satellite, regular TV, and VCR.
- A successful example of a similar product is the palm pilot, which evolved to include various features like cameras and MP3 players.

**2. Decisions Made**

- The remote control will be designed to be compatible with international markets.
- The production cost will be 50% of the selling price.
- The remote cont

In [10]:
#  Full JSON pipeline output
output = {
    'audio_file':          os.path.basename(AUDIO_PATH),
    'duration_seconds':    round(duration, 2),
    'num_speakers':        NUM_SPEAKERS,
    'raw_transcript':      raw_transcript,
    'labelled_transcript': labelled_transcript,
    'speaker_segments':    mapped_hyp,
    'summary':             summary,
}

json_path = f'{OUTPUT_DIR}/pipeline_output.json'
with open(json_path, 'w') as f:
    json.dump(output, f, indent=2)

print(' PIPELINE COMPLETE')

print(f'  {OUTPUT_DIR}/raw_transcript.txt')
print(f'  {OUTPUT_DIR}/diarized_transcript.txt')
print(f'  {OUTPUT_DIR}/meeting_summary.txt')
print(f'  {OUTPUT_DIR}/pipeline_output.json')


# Download outputs
from google.colab import files
for fname in ['diarized_transcript.txt', 'meeting_summary.txt', 'pipeline_output.json']:
    fpath = f'{OUTPUT_DIR}/{fname}'



 PIPELINE COMPLETE
  /content/output/raw_transcript.txt
  /content/output/diarized_transcript.txt
  /content/output/meeting_summary.txt
  /content/output/pipeline_output.json
